# Phase 8: Domain Adaptation Benchmark Comparison

Goals:
- Baseline vs DA performance comparison
- quantify DA improvement
- validate predicted vs actual transferability

In [14]:
import os
import sys
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Path settings
ROOT = r'..'
RESULTS_DIR = os.path.join(ROOT, 'results')
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
BENCH_RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(BENCH_RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 8: Domain Adaptation Benchmark Comparison")
print("="*100)
print(f"Results: {BENCH_RESULTS_DIR}\n")

PHASE 8: Domain Adaptation Benchmark Comparison
Results: ..\benchmark\results



In [15]:
# Load baseline and DA results
print("Loading baseline and DA results...\n")

baseline_results = {}
da_results = {}

properties = ['clearance', 'half_life', 'fu']

# Load baseline results (directly in RESULTS_DIR)
for prop in properties:
    baseline_path = os.path.join(RESULTS_DIR, f'01_baseline_{prop}.csv')
    if os.path.exists(baseline_path):
        df = pd.read_csv(baseline_path)
        baseline_results[prop] = df
        print(f"✓ Loaded baseline: {prop}")
    else:
        print(f"✗ Baseline not found: {prop}")

# Load DA results
da_dir = os.path.join(RESULTS_DIR, 'mmd_adaptation')
for prop in properties:
    da_path = os.path.join(da_dir, f'{prop}.csv')
    if os.path.exists(da_path):
        df = pd.read_csv(da_path)
        da_results[prop] = df
        print(f"✓ Loaded DA: {prop}")
    else:
        print(f"✗ DA not found: {prop}")

print()

Loading baseline and DA results...

✓ Loaded baseline: clearance
✓ Loaded baseline: half_life
✓ Loaded baseline: fu
✓ Loaded DA: clearance
✓ Loaded DA: half_life
✓ Loaded DA: fu



In [16]:
# Load transferability profile
print("Loading transferability profiles...\n")

try:
    with open(os.path.join(BENCH_RESULTS_DIR, 'phase_07_transferability_profile.json'), 'r') as f:
        transferability_profiles = json.load(f)
    print(f"✓ Loaded transferability profiles\n")
except:
    print(f"✗ Transferability profiles not found\n")
    transferability_profiles = {}

Loading transferability profiles...

✓ Loaded transferability profiles



In [17]:
# DA Benchmark Comparison
benchmark_results = {}

for prop in properties:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    if prop not in baseline_results or prop not in da_results:
        print(f"  Missing results (baseline or DA)")
        continue
    
    baseline_df = baseline_results[prop]
    da_df = da_results[prop]
    
    print(f"\n  Baseline results (Target-only):")
    print(f"    n_folds: {len(baseline_df)}")
    
    baseline_metrics = {}
    for metric in ['r', 'rho', 'r2', 'rmse', 'mae', 'mbe']:
        if metric in baseline_df.columns:
            mean = baseline_df[metric].mean()
            std = baseline_df[metric].std()
            baseline_metrics[metric] = {'mean': mean, 'std': std}
            print(f"    {metric}: {mean:.4f} ± {std:.4f}")
    
    print(f"\n  DA results (MMD-based):")
    print(f"    n_folds: {len(da_df)}")
    
    da_metrics = {}
    for metric in ['r', 'rho', 'r2', 'rmse', 'mae', 'mbe']:
        if metric in da_df.columns:
            mean = da_df[metric].mean()
            std = da_df[metric].std()
            da_metrics[metric] = {'mean': mean, 'std': std}
            print(f"    {metric}: {mean:.4f} ± {std:.4f}")
    
    # Compute improvements
    print(f"\n  DA Improvements (vs Baseline):")
    
    improvements = {}
    
    # Higher metrics (r, rho, r2) benefit from increase
    for metric in ['r', 'rho', 'r2']:
        if metric in baseline_metrics and metric in da_metrics:
            baseline_mean = baseline_metrics[metric]['mean']
            da_mean = da_metrics[metric]['mean']
            
            if baseline_mean > 0:
                improvement_pct = 100 * (da_mean - baseline_mean) / baseline_mean
            else:
                improvement_pct = 100 * (da_mean - baseline_mean)
            
            improvements[metric] = improvement_pct
            sign = '+' if improvement_pct > 0 else ''
            print(f"    {metric}: {baseline_mean:.4f} → {da_mean:.4f} ({sign}{improvement_pct:.1f}%)")
    
    # Lower metrics (rmse, mae, mbe) benefit from decrease
    for metric in ['rmse', 'mae']:
        if metric in baseline_metrics and metric in da_metrics:
            baseline_mean = baseline_metrics[metric]['mean']
            da_mean = da_metrics[metric]['mean']
            
            if baseline_mean > 0:
                improvement_pct = -100 * (da_mean - baseline_mean) / baseline_mean
            else:
                improvement_pct = -100 * (da_mean - baseline_mean)
            
            improvements[metric] = improvement_pct
            sign = '+' if improvement_pct > 0 else ''
            print(f"    {metric}: {baseline_mean:.4f} → {da_mean:.4f} ({sign}{improvement_pct:.1f}%)")
    
    # Compute overall DA improvement score
    r2_improvement = improvements.get('r2', 0)
    rmse_improvement = improvements.get('rmse', 0)
    
    overall_improvement = (r2_improvement + rmse_improvement) / 2
    
    print(f"\n  Overall DA Improvement: {overall_improvement:.1f}%")
    
    # Compare with predicted transferability
    if prop in transferability_profiles:
        profile = transferability_profiles[prop]
        predicted_success_prob = profile.get('da_success_probability', None)
        predicted_score = profile.get('overall_score', None)
        
        # Empirical success: positive improvement in r2
        actual_success = r2_improvement > 5
        
        print(f"\n  Prediction vs Actual:")
        if predicted_success_prob:
            print(f"    Predicted success prob: {predicted_success_prob:.0%}")
        if predicted_score:
            print(f"    Predicted score: {predicted_score}/100")
        print(f"    Actual improvement: {overall_improvement:.1f}%")
        print(f"    Actual success (>5% improvement): {'YES' if actual_success else 'NO'}")
        
        benchmark_results[prop] = {
            'baseline': baseline_metrics,
            'da': da_metrics,
            'improvements': improvements,
            'overall_improvement_pct': float(overall_improvement),
            'predicted_success_prob': float(predicted_success_prob) if predicted_success_prob else None,
            'predicted_score': int(predicted_score) if predicted_score else None,
            'actual_improvement_pct': float(r2_improvement),
            'actual_success': bool(actual_success),
        }


Property: CLEARANCE

  Baseline results (Target-only):
    n_folds: 5
    r2: 0.0519 ± 0.0805
    rmse: 0.9666 ± 0.0576
    mae: 0.7104 ± 0.0279
    mbe: -0.0156 ± 0.1488

  DA results (MMD-based):
    n_folds: 5
    r: 0.3592 ± 0.0703
    rho: 0.3736 ± 0.0772
    r2: 0.0538 ± 0.0790
    rmse: 0.9666 ± 0.0730
    mae: 0.7195 ± 0.0327
    mbe: -0.0646 ± 0.0518

  DA Improvements (vs Baseline):
    r2: 0.0519 → 0.0538 (+3.6%)
    rmse: 0.9666 → 0.9666 (+0.0%)
    mae: 0.7104 → 0.7195 (-1.3%)

  Overall DA Improvement: 1.8%

  Prediction vs Actual:
    Predicted success prob: 20%
    Predicted score: 44/100
    Actual improvement: 1.8%
    Actual success (>5% improvement): NO

Property: HALF_LIFE

  Baseline results (Target-only):
    n_folds: 5
    r2: 0.0458 ± 0.0678
    rmse: 0.9695 ± 0.0760
    mae: 0.7275 ± 0.0569
    mbe: -0.0840 ± 0.1840

  DA results (MMD-based):
    n_folds: 5
    r: 0.3713 ± 0.0654
    rho: 0.3911 ± 0.0329
    r2: -0.0015 ± 0.1770
    rmse: 0.9885 ± 0.0809
    

In [18]:
# Summary Report
print(f"\n\n{'='*100}")
print("DA BENCHMARK SUMMARY")
print(f"{'='*100}\n")

summary_data = []

for prop, result in benchmark_results.items():
    baseline_r2 = result['baseline'].get('r2', {}).get('mean', 0)
    da_r2 = result['da'].get('r2', {}).get('mean', 0)
    
    row = {
        'Property': prop,
        'Baseline_R2': f"{baseline_r2:.4f}",
        'DA_R2': f"{da_r2:.4f}",
        'R2_Improvement': f"{result['actual_improvement_pct']:.1f}%",
        'Overall_Improvement': f"{result['overall_improvement_pct']:.1f}%",
        'Success': 'YES' if result['actual_success'] else 'NO',
    }
    summary_data.append(row)

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
else:
    print("No comparison results available")
print()



DA BENCHMARK SUMMARY

 Property Baseline_R2   DA_R2 R2_Improvement Overall_Improvement Success
clearance      0.0519  0.0538           3.6%                1.8%      NO
half_life      0.0458 -0.0015        -103.3%              -52.6%      NO
       fu      0.1338  0.1473          10.1%                5.6%     YES



In [19]:
# Cross-validation analysis
print(f"\n{'='*100}")
print("FOLD-BY-FOLD ANALYSIS")
print(f"{'='*100}\n")

for prop in properties:
    if prop not in baseline_results or prop not in da_results:
        continue
    
    baseline_df = baseline_results[prop].copy()
    da_df = da_results[prop].copy()
    
    print(f"\n{prop.upper()}:")
    print(f"  {'Fold':<6} {'Baseline R2':<15} {'DA R2':<15} {'Improvement':<15}")
    print(f"  {'-'*50}")
    
    for fold_idx in range(len(baseline_df)):
        baseline_r2 = baseline_df.iloc[fold_idx]['r2'] if 'r2' in baseline_df.columns else 0
        da_r2 = da_df.iloc[fold_idx]['r2'] if 'r2' in da_df.columns else 0
        
        if baseline_r2 > 0:
            improvement = 100 * (da_r2 - baseline_r2) / baseline_r2
        else:
            improvement = 100 * (da_r2 - baseline_r2)
        
        sign = '+' if improvement > 0 else ''
        print(f"  {fold_idx:<6} {baseline_r2:<15.4f} {da_r2:<15.4f} {sign}{improvement:>13.1f}%")


FOLD-BY-FOLD ANALYSIS


CLEARANCE:
  Fold   Baseline R2     DA R2           Improvement    
  --------------------------------------------------
  0      0.0097          -0.0111                -213.7%
  1      -0.0510         -0.0500         +          0.1%
  2      0.0488          0.0934          +         91.3%
  3      0.0901          0.1120          +         24.3%
  4      0.1619          0.1246                  -23.1%

HALF_LIFE:
  Fold   Baseline R2     DA R2           Improvement    
  --------------------------------------------------
  0      0.1029          0.1266          +         23.0%
  1      0.0153          -0.0883                -678.8%
  2      0.1328          0.1949          +         46.8%
  3      -0.0027         0.0112          +          1.4%
  4      -0.0194         -0.2520                 -23.3%

FU:
  Fold   Baseline R2     DA R2           Improvement    
  --------------------------------------------------
  0      0.2028          0.1865                   -

In [20]:
# Save results
def convert_to_python_types(obj):
    if isinstance(obj, dict):
        return {k: convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj) if isinstance(obj, np.floating) else int(obj)
    elif isinstance(obj, (pd.Timestamp, np.datetime64)):
        return str(obj)
    return obj

with open(os.path.join(BENCH_RESULTS_DIR, 'phase_08_da_benchmark.json'), 'w') as f:
    json.dump(convert_to_python_types(benchmark_results), f, indent=2)

print(f"\n\n✓ Results saved to benchmark/results/")
print(f"  - phase_08_da_benchmark.json")
print("\nPhase 8 Complete! Baseline vs DA Comparison Finished.")



✓ Results saved to benchmark/results/
  - phase_08_da_benchmark.json

Phase 8 Complete! Baseline vs DA Comparison Finished.
